# Tine step 2: pool the coding datasets and rerun the CPU evaluation

Use a **CPU runtime**. Run all cells (Runtime → Run all). Takes roughly 10–20 minutes.

Needs the Drive folder from the first notebook (`MyDrive/tine`, with the benchmark archive
already extracted). Nothing is downloaded from Hugging Face, no model is called, no GPU is used.

What it does:
1. Imports the graded HumanEval, MBPP and SWE-bench Verified results from the archive you already have.
2. Pools them with LiveCodeBench into one `coding` group (about 2,000 tasks instead of 1,055),
   keeping only models present in every dataset and dropping models with no recorded cost.
3. Runs the evaluation twice: against `gpt-5` (the most accurate model, the honest "best single")
   and against `claude-sonnet-4` (your earlier choice).
4. Prints one short results block. Copy that block back to Claude.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path

BRANCH = "claude/gifted-gauss-zbjna8"  # change to "main" after this branch is merged
ROOT = Path("/content/drive/MyDrive/tine")
REPO = Path("/content/tine")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/Nandu-Dammalapti/tine.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
os.environ.update(MPLBACKEND="Agg", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 1. Import the other graded coding datasets from the archive already on Drive.
manifest = json.loads((ROOT / "benchmark" / "manifest.json").read_text())
BENCH = Path(manifest["bench_dir"])
inputs = [ROOT / "outcomes.jsonl"]  # LiveCodeBench, imported in the first notebook
import_errors = {}
for name in ["humaneval", "mbpp", "swe-bench"]:
    out = ROOT / f"outcomes-{name}.jsonl"
    if not out.exists():
        result = subprocess.run([sys.executable, "-m", "tine", "import-llmrouterbench",
                                 "--input", str(BENCH / name), "--output", str(out),
                                 "--tenant", "benchmark", "--dataset", name,
                                 "--run-selection", "latest"], capture_output=True, text=True)
        if result.returncode != 0:
            import_errors[name] = result.stderr.strip().splitlines()[-1]
            print(f"SKIPPED {name}: {import_errors[name]}")
            continue
    inputs.append(out)
    print("ready:", out.name)

In [ ]:
# 2. Pool into one "coding" group. "openrouter" is itself a router, not a model, so it is excluded.
POOLED = ROOT / "coding-outcomes.jsonl"
combine = subprocess.run([sys.executable, "-m", "tine", "combine-outcomes",
                          "--input", *map(str, inputs), "--output", str(POOLED),
                          "--task-type", "coding", "--exclude-model", "openrouter"],
                         capture_output=True, text=True)
if combine.returncode != 0:
    raise RuntimeError(combine.stderr)
combine_report = json.loads(combine.stdout)
print(json.dumps({k: combine_report[k] for k in ("rows", "tasks_by_source", "models_dropped")}, indent=2))

In [ ]:
# 3. Preflight and CPU evaluation for each frontier.
from tine.data import read_outcomes
from tine.pipeline import preflight

pooled = read_outcomes(POOLED)
FRONTIERS = [m for m in ["gpt-5", "claude-sonnet-4"] if m in set(pooled.model)]
readiness = {}
for frontier in FRONTIERS:
    readiness[frontier] = preflight(pooled, frontier, epsilon=0.02, delta=0.05, seed=42)
    out = ROOT / f"coding-lexical-{frontier}"
    subprocess.run([sys.executable, "-m", "tine", "run", "--input", str(POOLED),
                    "--frontier", frontier, "--output", str(out),
                    "--epsilon", "0.02", "--delta", "0.05", "--seed", "42"],
                   check=True, capture_output=True, text=True)
    print("done:", out.name)

In [ ]:
# 4. Copy everything this cell prints back to Claude.
import pandas as pd
pd.set_option("display.width", 200)

acc = pooled.assign(ok=pooled.success.astype(bool)).groupby("model").agg(
    accuracy=("ok", "mean"), mean_cost_usd=("cost_usd", "mean"))
print("POOLED MODELS (top 15 by accuracy)")
print(acc.sort_values("accuracy", ascending=False).head(15).round(4).to_string())
print("\nIMPORT ERRORS:", import_errors or "none")
print("TASKS:", combine_report["tasks_by_source"])
print("DROPPED:", combine_report["models_dropped"])
for frontier in FRONTIERS:
    out = ROOT / f"coding-lexical-{frontier}"
    r = readiness[frontier]
    print(f"\n===== frontier = {frontier}")
    print("risk-calibration tasks:", [g["risk_calibration"] for g in r["groups"]],
          "| minimum needed:", r["minimum_risk_tasks_at_zero_degradations"])
    for c in json.loads((out / "calibration.json").read_text()):
        stop = next((x for x in c["candidates"] if x["tested"] and not x["certified"]), None)
        line = f"  {c['method']:9s} tau={None if c['tau'] is None else round(c['tau'], 2)} (n={c['n']}, {c['reason']})"
        if c["tau"] is not None:
            line += f" degradations={c['degradations']} bound={c['upper_bound']:.4f}"
        if stop:
            line += (f" | stopped at tau={stop['tau']:.2f}: {stop['degradations']} degradations,"
                     f" bound {stop['upper_bound']:.4f}")
        print(line)
    m = pd.read_csv(out / "metrics.csv")
    cols = ["method", "n", "accuracy", "frontier_accuracy", "quality_loss",
            "degradation_rate", "savings_fraction", "oracle_gap_closed_fraction", "frontier_count"]
    print(m[cols].round(4).to_string(index=False))

Done. Copy the printed block above, then choose **Runtime → Disconnect and delete runtime**.
Reports are saved on Drive under `MyDrive/tine/coding-lexical-*`.